In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pickle

In [2]:
with open('../../output/models/xgboost/hyperparam-first-cog.pkl', 'rb+') as f:
    hyperparam_cog = pickle.load(f)
with open('../../output/models/xgboost/hyperparam-first-delta-dif.pkl', 'rb+') as f:
    hyperparam_delta_dif = pickle.load(f)
with open('../../output/models/xgboost/hyperparam-first-dif.pkl', 'rb+') as f:
    hyperparam_dif = pickle.load(f)

In [3]:
HYPERPARAMS = [
    'n_estimators',
    'max_depth',
    'eta',
    'subsample',
    # 'colsample_bytree',
    # 'gamma',
    # 'min_child_weight'
]

df_cog = pd.DataFrame(hyperparam_cog).melt()
df_cog.columns = HYPERPARAMS + ['loss']
df_cog['var'] = 'delta_COG'

df_delta_dif = pd.DataFrame(hyperparam_delta_dif).melt()
df_delta_dif.columns = HYPERPARAMS + ['loss']
df_delta_dif['var'] = 'delta_dif'

df_dif = pd.DataFrame(hyperparam_dif).melt()
df_dif.columns = HYPERPARAMS + ['loss']
df_dif['var'] = 'dif'

In [4]:
df_delta_dif.iloc[df_delta_dif['loss'].argmin(), :]

n_estimators          128
max_depth               4
eta                  0.05
subsample             0.8
loss             0.004202
var             delta_dif
Name: 49, dtype: object

In [5]:
df_dif.iloc[df_dif['loss'].argmin(), :]

n_estimators         128
max_depth              4
eta                 0.05
subsample            1.0
loss            0.004092
var                  dif
Name: 50, dtype: object

In [8]:
df_cog.iloc[df_cog['loss'].argmin(), :]

n_estimators          128
max_depth               4
eta                  0.05
subsample             0.8
loss             0.008146
var             delta_COG
Name: 49, dtype: object

In [7]:
grouped_mean = df_cog.groupby(
    ['n_estimators', 'max_depth', 'eta', 'subsample']
)['loss'].mean()
grouped_std = df_cog.groupby(
    ['n_estimators', 'max_depth', 'eta', 'subsample']
)['loss'].std()

grouped_mean.index[grouped_std.argmin()]

/tmp/ipykernel_21891/2878022533.py:8: FutureWarning: The behavior of Series.argmax/argmin with skipna=False and NAs, or with all-NAs is deprecated. In a future version this will raise ValueError.
  grouped_mean.index[grouped_std.argmin()]


(np.int64(128), np.int64(8), np.float64(0.05), np.float64(1.0))

In [9]:
df_cog_subsample_eq_1 = df_cog[
    (df_cog['subsample'] == 1.0) &
    (df_cog['max_depth'] < 6)
]
df_cog_subsample_eq_1.iloc[ df_cog_subsample_eq_1['loss'].argmin(), : ]

n_estimators          128
max_depth               4
eta                  0.05
subsample             1.0
loss             0.007855
var             delta_COG
Name: 38, dtype: object

In [4]:
df = pd.concat([df_cog, df_delta_dif, df_dif])
df_hyper = df.groupby(by = HYPERPARAMS + ['var'])\
    .agg(['mean', 'std'])\
    ['loss']\
    .reset_index()

In [14]:
df_hyper[df_hyper['var'] == 'delta_COG']\
    .groupby('subsample')\
    ['mean']\
    .mean()

subsample
0.6    0.009961
0.8    0.009937
1.0    0.009933
Name: mean, dtype: float64

In [28]:
min_mean_var = df_hyper.groupby('var')\
    ['mean'].idxmin()\
    .tolist()

df_hyper.iloc[min_mean_var, :]

,n_estimators,max_depth,eta,subsample,colsample_bytree,gamma,min_child_weight,var,mean,std
5049,128,4,0.1,1.0,0.8,0.0,1,delta_COG,0.008205,0.000883
4891,128,4,0.1,0.6,0.8,0.0,3,delta_dif,0.004491,0.000316
5057,128,4,0.1,1.0,0.8,0.0,5,dif,0.004789,0.000562


In [ ]:
df_hyper.iloc[min_mean_var[0], :].to_dict()

{'n_estimators': 128,
 'max_depth': 4,
 'eta': 0.1,
 'subsample': 1.0,
 'colsample_bytree': 0.8,
 'gamma': 0.0,
 'min_child_weight': 1,
 'var': 'delta_COG',
 'mean': 0.00820536100977866,
 'std': 0.0008832989052061287}

In [33]:
df_hyper.iloc[min_mean_var[1], :].to_dict()

{'n_estimators': 128,
 'max_depth': 4,
 'eta': 0.1,
 'subsample': 0.6,
 'colsample_bytree': 0.8,
 'gamma': 0.0,
 'min_child_weight': 3,
 'var': 'delta_dif',
 'mean': 0.004490623666272559,
 'std': 0.00031626838823225426}

In [34]:
df_hyper.iloc[min_mean_var[2], :].to_dict()

{'n_estimators': 128,
 'max_depth': 4,
 'eta': 0.1,
 'subsample': 1.0,
 'colsample_bytree': 0.8,
 'gamma': 0.0,
 'min_child_weight': 5,
 'var': 'dif',
 'mean': 0.004788907445635161,
 'std': 0.0005624853096510878}